# E2 - MSA and tokenization

Manual Multi-Head Self-Attention vs PyTorch reference; row, patch and CNN-stem tokenization; attention visualization.

> **Default:** notebooks run in `SMOKE=True` mode so a fresh clone can validate structure without downloading large datasets. Switch to `SMOKE=False` for real experiments.

## Session 0 - Environment

In [ ]:
from pathlib import Path
import os, sys, torch
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
SMOKE = True
SEED = 42
print("repo:", ROOT)
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())

## Session 1 - Tokenizer shape audit

In [ ]:
import torch
from E2.src.tokenizers import RowTokenizer,PatchTokenizer,CNNStemTokenizer
x=torch.randn(2,1,28,28)
for t in [RowTokenizer(),PatchTokenizer(4),CNNStemTokenizer()]: print(type(t).__name__, t(x).shape)

## Session 2 - Manual MSA equation and unit check

In [ ]:
from E2.src.attention import ManualMultiHeadSelfAttention,TorchMultiHeadSelfAttention
z=torch.randn(2,17,64)
manual=ManualMultiHeadSelfAttention(64,4); ref=TorchMultiHeadSelfAttention(64,4)
print(manual(z).shape, ref(z).shape)

## Session 3 - Classifier and smoke training

In [ ]:
from torch.utils.data import DataLoader
from common.synthetic import classification_dataset
from common.seed import resolve_device, seed_everything
from common.engine import fit_classifier
from E2.src.models import build_e2_model
seed_everything(42); device=resolve_device(); ds=classification_dataset(32); dl=DataLoader(ds,batch_size=8,shuffle=True)
model=build_e2_model("patch4","manual",num_heads=4).to(device); crit=torch.nn.CrossEntropyLoss(); opt=torch.optim.AdamW(model.parameters(),lr=3e-4)
fit_classifier(model,dl,dl,opt,crit,device,1)[-1]

## Session 4 - Full experiment matrix
Run Manual/PyTorch MSA x 3 tokenizers, then heads 2/4/8 on the best tokenizer. Export a comparison table and attention heatmaps.